# Anomaly Detection Masterclass — CRISP-DM
**Kaggle/ULB Credit Card Fraud**

This project is organized around decisions, evidence and operational constraints rather than around algorithms.


## 1. Business Understanding
Fraud is a rare-event problem with asymmetric costs and finite investigator capacity.

**Quiz:** Can 99.8% accuracy be useful when fraud is only about 0.17% of observations?
**Answer:** Not necessarily. A trivial classifier can be extremely accurate while detecting no fraud.

**Success criteria:** PR-AUC, precision@budget, recall@budget, alert volume and stability.


## 2. Data Understanding
The benchmark has 284,807 transactions and 492 fraud cases. V1–V28 are anonymized PCA components; Time and Amount are interpretable fields.

**Quiz:** Should an unsupervised detector use Class during fitting?
**Answer:** No. Class is for offline evaluation unless the problem is explicitly reframed as supervised/semi-supervised.


In [ ]:
from src.data import load,prepare,validate,temporal_split
df=prepare(load())
validate(df)


In [ ]:
df.describe().T


In [ ]:
df["Class"].value_counts(normalize=True)


## 3. Data Preparation
We log-transform Amount and preserve temporal ordering. Scaling is fitted only on training data.

**Quiz:** Why not fit a scaler on all rows?
**Answer:** That leaks information from validation/test distributions into training.


In [ ]:
train,val,test=temporal_split(df)
features=[f"V{i}" for i in range(1,29)]+["Time","Amount_log1p"]
Xtr=train[features].to_numpy(); Xv=val[features].to_numpy(); yv=val.Class.to_numpy()
train.shape,val.shape,test.shape


## 4. Modeling
### MAD
A robust statistical baseline.

### Isolation Forest
Liu, Ting & Zhou (2008): anomalies are easier to isolate. Efficient subsampling is a key design point.

### LOF
Breunig et al. (2000): measures local density deviation.

### One-Class SVM
Learns a boundary around normal observations and provides a useful contrasting inductive bias.

**Quiz:** Why compare methods with different inductive biases?
**Answer:** Global isolation, local density and boundary-based methods can detect different anomaly geometries.


In [ ]:
from src.detectors import *
from sklearn.preprocessing import StandardScaler
rows=[]; models={}
_,s=iforest(Xtr,Xv); models["Isolation Forest"]=s; rows.append({"model":"Isolation Forest",**evaluate(yv,s)})
sc=StandardScaler().fit(Xtr)
_,s=lof(sc.transform(Xtr),sc.transform(Xv)); models["LOF"]=s; rows.append({"model":"LOF",**evaluate(yv,s)})
_,s=ocsvm(Xtr,Xv); models["One-Class SVM"]=s; rows.append({"model":"One-Class SVM",**evaluate(yv,s)})
s=mad_score(Xtr,Xv); models["MAD baseline"]=s; rows.append({"model":"MAD baseline",**evaluate(yv,s)})
pd.DataFrame(rows).sort_values("pr_auc",ascending=False)


## 5. Deep Learning Extension — Autoencoder
Train on normal transactions and use reconstruction error as an anomaly score.

Important risks: an overpowered network can reconstruct anomalies too well; bottleneck size, regularization and threshold selection matter.

**Quiz:** Does high reconstruction error mean fraud probability?
**Answer:** No. It is an anomaly score unless calibrated and validated for a specific operational interpretation.


## 6. Evaluation
PR-AUC is emphasized because fraud is rare. ROC-AUC is useful but should not be the sole criterion.

The operational question is: **what happens when investigators can review only 0.1%, 0.5%, 1% or 2% of transactions?**

**Quiz:** Should the test set determine the final threshold?
**Answer:** No. Use validation data and business capacity to select it; reserve the test set for final evaluation.


In [ ]:
ensemble=np.mean([rank01(s) for s in models.values()],axis=0)
evaluate(yv,ensemble,budget=.01)


## 7. AutoResearch — Hill Climbing
`src/autoresearch.py` explores Isolation Forest configurations using a reproducible local search.

Objective:
`0.60*PR-AUC + 0.25*recall@budget + 0.15*precision@budget - complexity_penalty`

Guardrails:
- temporal validation
- fixed seed
- explicit search space
- experiment ledger
- no test-set tuning

**Quiz:** Is hill climbing guaranteed to find the global optimum?
**Answer:** No. It is a local-search heuristic.


## 8. Deployment / Synthesis
Reference architecture:

transaction → validation → feature service → detector → policy/threshold → investigation queue → feedback → monitoring → retraining.

AI-engineering concerns:
training/serving parity, model versioning, drift, alert volume, feedback labels, latency, rollback, shadow deployment, audit logs and reproducibility.

### Final synthesis
1. Which detector wins at the chosen budget?
2. What false positives appear?
3. What evidence supports deployment?
4. What evidence says more research is needed?
5. What monitoring catches failure early?
